# 02 — Derivação de Chave Criptográfica via Argon2id

**Objetivo:** transformar o perfil STR (matéria-prima com ~10–22 bits de
entropia) em material criptográfico de comprimento fixo (32 bytes,
compatível com AES-256) através de uma função de derivação memory-hard.

**Pipeline desta etapa:**
1. Regeneração dos perfis sintéticos (mesma semente = mesmos dados)
2. Representação canônica do perfil (serialização determinística)
3. Derivação de chave com Argon2id (salt aleatório)
4. Benchmark de tempo no hardware local
5. Verificações: perfis distintos → chaves distintas; salts distintos → chaves distintas

**Contexto de segurança:** a KDF não *cria* entropia — ela torna a busca
exaustiva (força bruta) computacionalmente custosa, "esticando" o custo
por tentativa via consumo de memória e CPU.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import hashlib
import secrets
import time

import numpy as np
import pandas as pd

from argon2.low_level import hash_secret_raw, Type
from src.entropia import entropia_shannon

rng = np.random.default_rng(seed=2024)  # MESMA semente do Notebook 01

print(f"Raiz do projeto: {RAIZ}")
print("Bibliotecas criptográficas carregadas com sucesso.")

Raiz do projeto: /home/ricardo/Projetos/tcc-genetica-cripto
Bibliotecas criptográficas carregadas com sucesso.


## 1. Regeneração dos perfis (reprodutibilidade)

Como usamos a MESMA semente (`seed=2024`) do Notebook 01, os 5000 perfis
regenerados aqui são **idênticos aos da análise anterior**. Este é o
mecanismo que garante que qualquer pessoa rodando o pipeline obtenha
exatamente os mesmos resultados — requisito de reprodutibilidade do artigo.

In [2]:
PAINEL_STR = {
    "D3S1358":  {15: 0.21, 16: 0.26, 17: 0.22, 18: 0.22, 19: 0.09},
    "TH01":     {6: 0.23, 7: 0.19, 8: 0.31, 9: 0.17, 9.3: 0.10},
    "D21S11":   {28: 0.16, 29: 0.24, 30: 0.27, 31: 0.18, 32: 0.15},
    "D18S51":   {13: 0.15, 14: 0.21, 15: 0.33, 16: 0.19, 17: 0.12},
    "D5S818":   {11: 0.36, 12: 0.38, 13: 0.20, 14: 0.06},
}

N_INDIVIDUOS = 5000

def sortear_individuo(painel: dict, rng) -> dict:
    perfil = {}
    for locus, freqs in painel.items():
        alelos, probs = zip(*freqs.items())
        perfil[locus] = tuple(rng.choice(list(map(str, alelos)), size=2, p=probs))
    return perfil

perfis = [sortear_individuo(PAINEL_STR, rng) for _ in range(N_INDIVIDUOS)]

print(f"Perfis regenerados: {len(perfis)}")
print(f"Sanidade — indivíduo 0, D3S1358: {perfis[0]['D3S1358']}")



Perfis regenerados: 5000
Sanidade — indivíduo 0, D3S1358: (np.str_('17'), np.str_('16'))


In [7]:
import math

def probs_genotipicas(freq_alelos: dict) -> dict:
    """Probabilidades dos genótipos canônicos de um locus sob Hardy-Weinberg."""
    probs = {}
    alelos = sorted(freq_alelos)
    for i, a in enumerate(alelos):
        for b in alelos[i:]:                      # (a,b) com a <= b — genótipo canônico
            p, q = freq_alelos[a], freq_alelos[b]
            probs[(a, b)] = p * q if a == b else 2 * p * q
    return probs

H_inf_total = 0.0
for locus, freqs in PAINEL_STR.items():
    probs = probs_genotipicas(freqs)
    p_max = max(probs.values())
    H_inf_locus = -math.log2(p_max)
    H_inf_total += H_inf_locus
    print(f"{locus}: H∞ = {H_inf_locus:.2f} bits (genótipo mais provável p = {p_max:.4f})")

print(f"\nH∞ do perfil completo (soma dos 5 loci): {H_inf_total:.2f} bits")
print(f"Referência: H_shannon canônica ≈ 17,74 bits | par ordenado ≈ 21,54 bits")

D3S1358: H∞ = 3.13 bits (genótipo mais provável p = 0.1144)
TH01: H∞ = 2.81 bits (genótipo mais provável p = 0.1426)
D21S11: H∞ = 2.95 bits (genótipo mais provável p = 0.1296)
D18S51: H∞ = 2.85 bits (genótipo mais provável p = 0.1386)
D5S818: H∞ = 1.87 bits (genótipo mais provável p = 0.2736)

H∞ do perfil completo (soma dos 5 loci): 13.61 bits
Referência: H_shannon canônica ≈ 17,74 bits | par ordenado ≈ 21,54 bits


## 2. Representação canônica do perfil

Antes de derivar a chave, o perfil genético precisa virar uma **sequência
de bytes determinística e sem ambiguidade**. Exigências:

1. **Ordem fixa de loci** — o mesmo dicionário em Python pode iterar em
   ordens diferentes em contextos distintos; fixamos a ordem alfabética
2. **Alelos ordenados** — o genótipo (16,17) é o mesmo que (17,16);
   sortear elimina essa ambiguidade
3. **Formato unívoco** — delimitadores que impedem "(1,15)" de ser lido
   como "(11,5)"

Analogia: é o "formato de registro" do mundo criptográfico — assim como
um arquivo DBF do Clipper tinha campos de tamanho fixo para nunca
ambiguar, aqui a canonização garante que bytes iguais ↔ perfil igual.

In [3]:
def perfil_canonico(perfil: dict) -> bytes:
    """
    Serializa o perfil STR em bytes canônicos.

    Regras: loci em ordem alfabética, alelos ordenados numericamente
    (quando decimais) e separador '|' entre campos.
    """
    partes = []
    for locus in sorted(perfil.keys()):
        a1, a2 = perfil[locus]
        # ordenação: tenta numérica (TH01 tem alelo 9.3), recorre à textual
        try:
            par_ordenado = sorted([a1, a2], key=float)
        except ValueError:
            par_ordenado = sorted([a1, a2])
        partes.append(f"{locus}:{par_ordenado[0]}:{par_ordenado[1]}")
    return "|".join(partes).encode("utf-8")


# Demonstração: heterozigoto em duas ordens → MESMOS bytes
p1 = {"TH01": ("9", "7"), "D21S11": ("30", "29")}
p2 = {"TH01": ("7", "9"), "D21S11": ("29", "30")}  # mesmos alelos, ordem trocada

c1, c2 = perfil_canonico(p1), perfil_canonico(p2)
print(f"Perfil 1 canonizado: {c1}")
print(f"Perfil 2 canonizado: {c2}")
print(f"São idênticos? {c1 == c2}  <-- TEM QUE SER True")

# E agora o indivíduo 0 do nosso dataset:
canon0 = perfil_canonico(perfis[0])
print(f"\nIndivíduo 0 ({len(canon0)} bytes): {canon0}")

Perfil 1 canonizado: b'D21S11:29:30|TH01:7:9'
Perfil 2 canonizado: b'D21S11:29:30|TH01:7:9'
São idênticos? True  <-- TEM QUE SER True

Indivíduo 0 (61 bytes): b'D18S51:13:14|D21S11:28:32|D3S1358:16:17|D5S818:11:11|TH01:7:9'


## 3. Derivação de chave com Argon2id

Agora aplicamos a KDF **Argon2id** (vencedora da Password Hashing
Competition, 2015). Propriedades:

- **Memory-hard**: consome um bloco de memória configurável por derivação,
  tornando ataques massivamente paralelos (GPU/ASIC) economicamente inviáveis
- **Salt aleatório**: garante que dois perfis iguais com salts diferentes
  produzam chaves sem relação entre si, e impede ataques de rainbow table
- **Saída de comprimento fixo**: 32 bytes, pronta para AES-256

**Modelo de segurança adotado:** perfil genético + salt → chave.
O salt precisa ser armazenado junto à chave (não é segredo); a chave
resultante NÃO permite recuperar o perfil (função de mão única).

In [4]:
# Parâmetros do Argon2id (calibrados para hardware modesto)
PARAMS = dict(
    time_cost=3,      # iterações (passes sobre a memória)
    memory_cost=65536,  # 64 MiB de memória por derivação
    parallelism=1,    # threads (1 = compatível com qualquer máquina)
    hash_len=32,      # saída: 32 bytes (AES-256)
    type=Type.ID,     # variante híbrida (anti side-channel + anti GPU)
)

def derivar_chave(perfil: dict, salt: bytes, params: dict = PARAMS) -> bytes:
    """Deriva uma chave de 32 bytes do perfil STR canônico via Argon2id."""
    material = perfil_canonico(perfil)
    return hash_secret_raw(
        secret=material,
        salt=salt,
        **params,
    )

# Salt aleatório de 16 bytes (padrão mínimo recomendado)
salt_demo = secrets.token_bytes(16)

t0 = time.perf_counter()
chave_individuo0 = derivar_chave(perfis[0], salt_demo)
tempo_derivacao = time.perf_counter() - t0

print(f"Chave derivada (32 bytes): {chave_individuo0.hex()}")
print(f"Tempo de derivação: {tempo_derivacao:.3f} s")

Chave derivada (32 bytes): 1257add574f50ae91bbe45c0b1a7fc342319a7a11ecefcc7fa508e4dcaeae06d
Tempo de derivação: 0.272 s


In [6]:
import secrets
import statistics
import time

# Teste 1: Benchmark em amostra dos 5000 perfis
N_BENCHMARK = 100
salt_fixo = secrets.token_bytes(16)

tempos = []
for i, perfil in enumerate(perfis[:N_BENCHMARK]):
    t0 = time.perf_counter()
    chave = derivar_chave(perfil, salt_fixo)
    tempos.append(time.perf_counter() - t0)

tempo_medio = sum(tempos) / len(tempos)
tempo_min = min(tempos)
tempo_max = max(tempos)
desvio_padrao = statistics.stdev(tempos)  # σ amostral real (n-1 no denominador)

print(f"Benchmark ({N_BENCHMARK} derivações):")
print(f"  Médio: {tempo_medio*1000:.1f} ms")
print(f"  Mín: {tempo_min*1000:.1f} ms | Máx: {tempo_max*1000:.1f} ms")
print(f"  Desvio padrão (σ): {desvio_padrao*1000:.1f} ms")

def hamming_bits(b1: bytes, b2: bytes) -> int:
    """Distância de Hamming em nível de BIT (XOR + popcount)."""
    return (int.from_bytes(b1, 'big') ^ int.from_bytes(b2, 'big')).bit_count()

# Teste 2: Perfis distintos, mesmo salt → chaves sem correlação
chave1 = derivar_chave(perfis[0], salt_fixo)
chave2 = derivar_chave(perfis[1], salt_fixo)
d_perfis = hamming_bits(chave1, chave2)
print(f"\nTeste de diferenciação (perfis distintos, mesmo salt):")
print(f"  Distância Hamming: {d_perfis}/256 bits ({d_perfis/256:.1%})")
print(f"  Esperado (aleatório puro): ~128/256 bits (~50%)")

# Teste 3: Mesmo perfil, salts distintos → chaves sem correlação
salt_outro = secrets.token_bytes(16)
chave_salt1 = derivar_chave(perfis[0], salt_fixo)
chave_salt2 = derivar_chave(perfis[0], salt_outro)
d_salts = hamming_bits(chave_salt1, chave_salt2)
print(f"\nTeste de salting (mesmo perfil, salts distintos):")
print(f"  Distância Hamming: {d_salts}/256 bits ({d_salts/256:.1%})")
print(f"  Esperado (aleatório puro): ~128/256 bits (~50%)")

Benchmark (100 derivações):
  Médio: 237.5 ms
  Mín: 221.9 ms | Máx: 270.0 ms
  Desvio padrão (σ): 13.0 ms

Teste de diferenciação (perfis distintos, mesmo salt):
  Distância Hamming: 112/256 bits (43.8%)
  Esperado (aleatório puro): ~128/256 bits (~50%)

Teste de salting (mesmo perfil, salts distintos):
  Distância Hamming: 134/256 bits (52.3%)
  Esperado (aleatório puro): ~128/256 bits (~50%)


## 4. Leitura dos resultados

### Benchmark de desempenho
- Tempo médio: **__ ms** (no geladeiravelha, hardware modesto)
- Implicação de segurança: cada tentativa de força bruta custa __ ms
  + 64 MiB de memória, inviabilizando paralelização massiva

### Propriedades criptográficas verificadas
1. ✅ **Diferenciação**: perfis distintos geram chaves com ~50% de diferença (comportamento aleatório esperado)
2. ✅ **Independência de salt**: mesmo perfil com salt diferente gera chave sem correlação
3. ⚠️ **Entropia da fonte**: ~10–22 bits reais vs. 256 bits da saída — a KDF não cria entropia, apenas amplia o custo computacional do ataque

### Discussão metodológica
A abordagem adotada permite transformar perfil genético em chave utilizável,
mas **não elimina a necessidade de fator adicional** (senha, salt secreto)
quando o objetivo é segurança > 128 bits. Este é o gap que justifica futuras
extensões do trabalho (ver seção 5).